# Unidades Deliberativas Verificáveis: primeiro protótipo

Hoje uma opinião do arquivo LDS é só uma string solta dentro de uma lista: não há nenhum jeito de verificar, a partir dela, quem disse aquilo, em que trecho da transcrição, ou com que confiança. Este notebook tenta um primeiro pipeline mínimo para atacar esse problema: segmentar a transcrição por participante, resolver quem é quem, e tentar anexar evidência a cada opinião. O resultado é reportado como é, incluindo o que não funcionou.

In [1]:
import json
from pathlib import Path

import pandas as pd

## Carregando os dados

In [2]:
with open("dataset/PublicHearingBR_LDS.jsonl") as f:
    lds_records = [json.loads(line) for line in f]

In [3]:
sample_hearing = lds_records[0]
print(f"audiência de exemplo: id={sample_hearing['id']}")
print(f"participantes: {len(sample_hearing['metadados']['envolvidos'])}")

audiência de exemplo: id=1
participantes: 8


## Segmentação de turnos de fala

In [4]:
import re

In [5]:
naive_turn_pattern = re.compile(
    r"(?:O\s+SR\.|A\s+SRA\.)\s*([A-ZÀ-Ü][^(\n]{0,60})\([^)]*\)\s*-\s"
)
naive_matches = naive_turn_pattern.findall(sample_hearing["transcricao"])
naive_speaker_names = sorted({m.strip() for m in naive_matches})
print(f"turnos encontrados (regex exige parênteses): {len(naive_matches)}")
print("nomes únicos:", naive_speaker_names)

turnos encontrados (regex exige parênteses): 53
nomes únicos: ['ADRIANA VENTURA', 'AFONSO HAMM', 'ARLINDO CHINAGLIA', 'BIA KICIS', 'FILIPE BARROS', 'FLORENTINO NETO', 'GENERAL GIRÃO', 'GILVAN DA FEDERAL', 'LUIZ PHILIPPE DE ORLEANS E BRAGANÇA', 'MARCEL VAN HATTEM', 'MARCELO MORAES', 'MARIO FRIAS', 'PRESIDENTE']


Alguns participantes listados em `envolvidos` não aparecem nessa lista de nomes. Vale checar um deles diretamente no texto da transcrição.

In [6]:
missing_person = "Michael Shellenberger"
first_name_upper = missing_person.split()[0].upper()
position = sample_hearing["transcricao"].upper().find(first_name_upper, 5000)
print(sample_hearing["transcricao"][position - 20 : position + 60])

laborador americano Michael Shellenberger.

Confesso que eu nem sei ao certo a o


O nome aparece, mas sem parênteses depois: `O SR. MICHAEL SHELLENBERGER- Muito obrigado...`. Participantes sem filiação partidária (convidados estrangeiros, especialistas) têm um cabeçalho de turno diferente, sem `(PARTIDO)`, e às vezes sem espaço antes do hífen. O regex original perde esses turnos inteiramente.

In [7]:
turn_header_pattern = re.compile(
    r"(?:O\s+SR\.|A\s+SRA\.)\s*([A-ZÀ-Ü][A-ZÀ-Ü\.\s]{0,60}?)\s*(?:\(([^)]*)\))?\s*-\s?"
)


def split_into_turns(transcript):
    matches = list(turn_header_pattern.finditer(transcript))
    turns = []
    for index, match in enumerate(matches):
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(transcript)
        turns.append(
            {
                "raw_name": match.group(1).strip(),
                "party_info": (match.group(2) or "").strip(),
                "speech": transcript[start:end].strip(),
            }
        )
    return turns

In [8]:
sample_turns = split_into_turns(sample_hearing["transcricao"])
print(f"turnos encontrados (regex corrigido): {len(sample_turns)}")
print("nomes únicos:", sorted({t['raw_name'] for t in sample_turns}))

turnos encontrados (regex corrigido): 65
nomes únicos: ['ADRIANA VENTURA', 'AFONSO HAMM', 'ARLINDO CHINAGLIA', 'BIA KICIS', 'DAVID ÁGAPE', 'ELI VIEIRA ARAUJO JÚNIOR', 'FILIPE BARROS', 'FLORENTINO NETO', 'GENERAL GIRÃO', 'GILVAN DA FEDERAL', 'GLENN EDWARD GREENWALD', 'LUIZ PHILIPPE DE ORLEANS E BRAGANÇA', 'MARCEL VAN HATTEM', 'MARCELO MORAES', 'MARIO FRIAS', 'MICHAEL SHELLENBERGER', 'PRESIDENTE']


Com o parêntese opcional, o número de turnos sobe e os convidados sem filiação partidária passam a aparecer. Isso é o oposto de otimismo: o primeiro regex parecia funcionar (achava turnos em todas as audiências), mas escondia um viés de cobertura que só aparece quando se olha nome por nome.

## Resolução de pessoa

In [9]:
import unicodedata


def strip_accents(text):
    return "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )


def normalize_name(name):
    return strip_accents(name).upper().strip()

Um turno como `O SR. PRESIDENTE(Lucas Redecker. Bloco/PSDB - RS)` tem o nome de verdade dentro do parêntese, antes do primeiro ponto, não no lugar do cargo.

In [10]:
def resolve_turn_name(turn):
    if "." in turn["party_info"]:
        return turn["party_info"].split(".")[0].strip()
    return turn["raw_name"]


for turn in sample_turns:
    turn["resolved_name"] = resolve_turn_name(turn)

Nomes como `Glenn Greenwald` (envolvidos) e `GLENN EDWARD GREENWALD` (transcrição, com nome do meio) ou `Eli Vieira Junior` e `ELI VIEIRA ARAUJO JÚNIOR` não batem por igualdade exata. Uma comparação por conjunto de palavras (uma lista de nomes sendo subconjunto da outra) tolera isso, mas exige cuidado com nomes de uma palavra só, que dariam falso positivo fácil demais.

In [11]:
def names_match(name_a, name_b):
    tokens_a = set(normalize_name(name_a).split())
    tokens_b = set(normalize_name(name_b).split())
    if len(tokens_a) <= 1 or len(tokens_b) <= 1:
        return tokens_a == tokens_b
    return tokens_a <= tokens_b or tokens_b <= tokens_a

In [12]:
def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()


def resolve_person_speech(pessoa, turns):
    matched_turns = [t for t in turns if names_match(pessoa["nome"], t["resolved_name"])]
    speech = normalize_whitespace(" ".join(t["speech"] for t in matched_turns))
    return matched_turns, speech

In [13]:
sample_envolvidos = sample_hearing["metadados"]["envolvidos"]
sample_person_speech = {}
resolved_count = 0

for pessoa in sample_envolvidos:
    matched_turns, speech = resolve_person_speech(pessoa, sample_turns)
    sample_person_speech[pessoa["nome"]] = speech
    if matched_turns:
        resolved_count += 1
    print(f"{pessoa['nome']}: {len(matched_turns)} turno(s), {len(speech)} caracteres de fala")

print(f"\npessoas resolvidas: {resolved_count}/{len(sample_envolvidos)}")

Michael Shellenberger: 2 turno(s), 6430 caracteres de fala
Glenn Greenwald: 4 turno(s), 15733 caracteres de fala
Marcel van Hattem: 3 turno(s), 23350 caracteres de fala
Arlindo Chinaglia: 5 turno(s), 6558 caracteres de fala
Filipe Barros: 2 turno(s), 2471 caracteres de fala
Florentino Neto: 1 turno(s), 1877 caracteres de fala
Eli Vieira Junior: 2 turno(s), 11238 caracteres de fala
David Ágape: 4 turno(s), 17233 caracteres de fala

pessoas resolvidas: 8/8


Nesta audiência, todos os 8 participantes listados tiveram pelo menos um turno de fala encontrado. Isso não garante que a heurística de nomes generalize bem para as demais 205 audiências, só que funcionou aqui.

## Busca de evidência por citação direta

In [14]:
quote_pattern = re.compile(r'“([^”]{10,})”|"([^"]{10,})"')


def extract_quote(opinion_text):
    match = quote_pattern.search(opinion_text)
    if not match:
        return None
    return normalize_whitespace(match.group(1) or match.group(2))

In [15]:
def find_evidence(quote, person_speech):
    words = quote.split()
    for prefix_length in (10, 6, 4, 3):
        prefix = " ".join(words[:prefix_length])
        if len(prefix) > 5 and prefix in person_speech:
            return prefix
    return None

In [16]:
sample_udv_status = []

for pessoa in sample_envolvidos:
    person_speech = sample_person_speech[pessoa["nome"]]
    for opinion_text in pessoa["opinioes"]:
        quote = extract_quote(opinion_text)
        if quote is None:
            sample_udv_status.append("no_quote")
        elif find_evidence(quote, person_speech):
            sample_udv_status.append("quote_found")
        else:
            sample_udv_status.append("quote_not_found")

In [17]:
from collections import Counter

print(Counter(sample_udv_status))

Counter({'no_quote': 17, 'quote_not_found': 5, 'quote_found': 4})


A maioria das opiniões nem tem citação direta (é paráfrase da matéria), e mesmo entre as que têm, boa parte não é localizável na fala da própria pessoa por correspondência literal de prefixo. O resultado é modesto, não bom: buscar evidência por texto exato tem alcance limitado.

## Montagem da UDV v0

In [18]:
def build_udv(hearing_id, pessoa, opinion_text, person_speech):
    quote = extract_quote(opinion_text)
    if quote is None:
        evidence, status = [], "no_quote"
    else:
        found = find_evidence(quote, person_speech)
        evidence = [{"text": found}] if found else []
        status = "quote_found" if found else "quote_not_found"

    return {
        "hearing_id": hearing_id,
        "actor": {"name": pessoa["nome"], "role": pessoa["cargo"]},
        "proposition": opinion_text,
        "evidence": evidence,
        "stance": None,
        "speech_act": None,
        "status": status,
    }

In [19]:
sample_udvs = [
    build_udv(sample_hearing["id"], pessoa, opinion_text, sample_person_speech[pessoa["nome"]])
    for pessoa in sample_envolvidos
    for opinion_text in pessoa["opinioes"]
]

for udv in sample_udvs:
    if udv["status"] == "quote_found":
        print(udv)
        break

{'hearing_id': 1, 'actor': {'name': 'Arlindo Chinaglia', 'role': 'Deputado (PT-SP)'}, 'proposition': 'Apontou que, de 2012 a 2021, na soma das solicitações de remoção por país recebidas pela plataforma X, o Brasil figura em antepenúltimo da lista: "Portanto, tentar caracterizar que, no Brasil de hoje, e desde 2012, haja uma atitude de censura é um exagero retórico que se transforma em mentira."', 'evidence': [{'text': 'Portanto, tentar caracterizar'}], 'stance': None, 'speech_act': None, 'status': 'quote_found'}


In [20]:
for udv in sample_udvs:
    if udv["status"] == "no_quote":
        print(udv)
        break

{'hearing_id': 1, 'actor': {'name': 'Michael Shellenberger', 'role': 'Jornalista norte-americano'}, 'proposition': 'Acusou Alexandre de Moraes de censura ao solicitar o bloqueio de contas na rede social X.', 'evidence': [], 'stance': None, 'speech_act': None, 'status': 'no_quote'}


## Cobertura numa amostra maior

In [21]:
sample_size = 20
coverage_counter = Counter()
people_total = 0
people_resolved = 0

for hearing in lds_records[:sample_size]:
    turns = split_into_turns(hearing["transcricao"])
    for turn in turns:
        turn["resolved_name"] = resolve_turn_name(turn)

    for pessoa in hearing["metadados"]["envolvidos"]:
        people_total += 1
        matched_turns, person_speech = resolve_person_speech(pessoa, turns)
        if matched_turns:
            people_resolved += 1

        for opinion_text in pessoa["opinioes"]:
            udv = build_udv(hearing["id"], pessoa, opinion_text, person_speech)
            if not matched_turns:
                coverage_counter["person_not_resolved"] += 1
            else:
                coverage_counter[udv["status"]] += 1

In [22]:
print(f"pessoas resolvidas: {people_resolved}/{people_total}")
pd.Series(coverage_counter, name="opinioes").sort_values(ascending=False)

pessoas resolvidas: 101/108


no_quote               151
quote_not_found         71
quote_found             32
person_not_resolved     14
Name: opinioes, dtype: int64

## Conclusões

A resolução de pessoa funcionou bem na amostra de 20 audiências, mas só depois de corrigir o regex de turno para aceitar cabeçalhos sem parênteses, sem essa correção, convidados sem filiação partidária ficariam de fora silenciosamente.

A busca de evidência por citação direta é o ponto fraco deste v0: a maior parte das opiniões não tem citação (é paráfrase da matéria), e mesmo as que têm citação só são localizadas numa minoria dos casos, mesmo restringindo a busca à fala da própria pessoa e tolerando prefixos curtos. Não é um resultado bom, é honesto.

Para uma próxima versão, casar texto exato não é suficiente: opiniões parafraseadas exigem alguma forma de busca por similaridade semântica (não só léxica) contra a fala da pessoa, para virar evidência de verdade em vez de só um rótulo `no_quote`.